# IMDb Movie Review Opinion Mining — Jupyter Dashboard
**Case study:** Text cleaning, Bag-of-Words, TF-IDF, SVD/LSA, binary sentiment classification, enriched tags, and interactive dashboard.  
**Dataset:** *Sentiment Labelled Sentences*, IMDb subset (UCI, DOI: 10.24432/C57604).  
**Important:** This dataset contains 500 positive and 500 negative sentences; it contains **no ground-truth neutral class**.

Run **Kernel → Restart & Run All** after installing packages. The last cell opens a dashboard **inside Jupyter Notebook**. Figures and data are also exported to `outputs/` for your report.

## 0. Install dependencies (run once)

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn nltk wordcloud ipywidgets ipython

## 1. Imports and load the official IMDb subset

In [ ]:
from pathlib import Path
import io, re, json, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
from ipywidgets import widgets
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from nltk.stem import SnowballStemmer
from wordcloud import WordCloud

ROOT = Path.cwd()
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
ZIP = ROOT / 'sentiment+labelled+sentences.zip'
URL = 'https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip'
if not ZIP.exists():
    print('Attempting to download official UCI dataset...')
    try:
        urllib.request.urlretrieve(URL, ZIP)
    except Exception as exc:
        raise FileNotFoundError('Download the ZIP from https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences and put it next to this notebook; then rerun.') from exc

with zipfile.ZipFile(ZIP) as archive:
    imdb_member = next((x for x in archive.namelist() if x.endswith('imdb_labelled.txt')), None)
    if not imdb_member:
        raise ValueError('IMDb file not found inside ZIP.')
    text_data = archive.read(imdb_member).decode('utf-8-sig', errors='replace')
records = []
for line in text_data.splitlines():
    if '\t' not in line: continue
    review, label = line.rsplit('\t', 1)
    if label.strip() in ('0', '1') and review.strip():
        records.append({'review': review.strip(), 'label': int(label.strip())})
df = pd.DataFrame(records)
df['sentiment'] = df['label'].map({0:'Negative', 1:'Positive'})
print('IMDb reviews loaded:',len(df))
print('Missing values:',df.isna().sum().to_dict())
print('Duplicate rows:',int(df.duplicated().sum()))
display(df.head(10))

## 2. Clean text — retain negation words

In [ ]:
stemmer = SnowballStemmer('english')
NEGATIONS = {'no', 'nor', 'not', 'never'}
STOP = set(ENGLISH_STOP_WORDS) - NEGATIONS

def clean_text(text):
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", str(text).lower())
    # Convert common negation contractions so their polarity is not lost
    tokens = [re.sub(r"n't$", ' not', t) for t in tokens]
    tokens = [word for part in tokens for word in part.split()]
    return ' '.join(stemmer.stem(w) for w in tokens if len(w)>1 and w not in STOP)

df['clean_text'] = df.review.apply(clean_text)
removed = int(df.clean_text.eq('').sum())
df = df[df.clean_text.ne('')].reset_index(drop=True)
print('Removed empty cleaned reviews:', removed)
display(df[['review','clean_text','sentiment']].head(10))

## 3. Dataset overview

In [ ]:
counts = df.sentiment.value_counts().reindex(['Negative','Positive'],fill_value=0)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(counts.index,counts.values,color=['#c85b62','#2b9b86'])
ax.set(title='IMDb review sentiment distribution',ylabel='Number of reviews')
for i,v in enumerate(counts.values): ax.text(i,v+3,str(v),ha='center')
plt.tight_layout();fig.savefig(OUT/'01_distribution.png',dpi=180);display(fig);plt.close(fig)

## 4. Bag-of-Words and word frequencies (descriptive corpus analysis)

In [ ]:
bow=CountVectorizer(max_features=2000)
B=bow.fit_transform(df.clean_text)
terms=np.array(bow.get_feature_names_out())
frequencies=np.asarray(B.sum(axis=0)).ravel()
bow_df=pd.DataFrame({'term':terms,'frequency':frequencies}).sort_values('frequency',ascending=False).reset_index(drop=True)
print('BoW matrix shape:',B.shape)
display(bow_df.head(20))
fig,ax=plt.subplots(figsize=(9,6));top=bow_df.head(20).iloc[::-1]
ax.barh(top.term,top.frequency,color='#537ca3');ax.set(title='Top 20 Bag-of-Words terms',xlabel='Corpus frequency')
plt.tight_layout();fig.savefig(OUT/'02_bow.png',dpi=180);display(fig);plt.close(fig)

## 5. TF-IDF features (descriptive corpus analysis)

In [ ]:
tfidf_view=TfidfVectorizer(max_features=2000,ngram_range=(1,2),sublinear_tf=True)
Xview=tfidf_view.fit_transform(df.clean_text)
names=np.array(tfidf_view.get_feature_names_out())
means=np.asarray(Xview.mean(axis=0)).ravel()
tfidf_df=pd.DataFrame({'term':names,'mean_tfidf':means}).sort_values('mean_tfidf',ascending=False).reset_index(drop=True)
print('Descriptive TF-IDF matrix shape:',Xview.shape)
display(tfidf_df.head(20))
fig,ax=plt.subplots(figsize=(9,6));top=tfidf_df.head(20).iloc[::-1]
ax.barh(top.term,top.mean_tfidf,color='#8562a9');ax.set(title='Top 20 TF-IDF features',xlabel='Mean TF-IDF weight')
plt.tight_layout();fig.savefig(OUT/'03_tfidf.png',dpi=180);display(fig);plt.close(fig)

## 6. Dimensionality reduction using Truncated SVD

In [ ]:
svd=TruncatedSVD(n_components=2,random_state=42)
coords=svd.fit_transform(Xview)
df[['LSA_1','LSA_2']]=coords
explained=float(svd.explained_variance_ratio_.sum())
print('Original matrix:',Xview.shape,'Reduced matrix:',coords.shape)
print('2D explained variance:',round(explained*100,2),'%')
fig,ax=plt.subplots(figsize=(9,5))
for label,color in [(0,'#c85b62'),(1,'#2b9b86')]:
    sub=df.label.eq(label)
    ax.scatter(coords[sub,0],coords[sub,1],s=15,alpha=.43,c=color,label='Negative' if label==0 else 'Positive')
ax.set(title='Truncated SVD: IMDb reviews (true sentiment)',xlabel='Component 1',ylabel='Component 2')
ax.legend();plt.tight_layout();fig.savefig(OUT/'04_svd.png',dpi=180);display(fig);plt.close(fig)
print('Note: two components retain only part of total text variation.')

## 7. Train and evaluate a supervised model — no train/test leakage

In [ ]:
train_ids,test_ids=train_test_split(np.arange(len(df)),test_size=.20,stratify=df.label,random_state=42)
vectorizer=TfidfVectorizer(max_features=2000,ngram_range=(1,2),sublinear_tf=True)
Xtr=vectorizer.fit_transform(df.loc[train_ids,'clean_text'])
Xte=vectorizer.transform(df.loc[test_ids,'clean_text'])
ytr=df.loc[train_ids,'label']; yte=df.loc[test_ids,'label']
classifier=LogisticRegression(max_iter=2000,random_state=42)
classifier.fit(Xtr,ytr)
pred=classifier.predict(Xte)
acc=float(accuracy_score(yte,pred))
report=classification_report(yte,pred,labels=[0,1],target_names=['Negative','Positive'],output_dict=True,zero_division=0)
print('Training size:',len(train_ids),'Test size:',len(test_ids))
print('Test accuracy:',f'{acc:.2%}')
print(classification_report(yte,pred,labels=[0,1],target_names=['Negative','Positive'],zero_division=0))
fig,ax=plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(yte,pred,labels=[0,1],display_labels=['Negative','Positive'],cmap='Blues',ax=ax,colorbar=False)
ax.set_title('Confusion matrix (held-out 20% test set)')
plt.tight_layout();fig.savefig(OUT/'05_confusion.png',dpi=180);display(fig);plt.close(fig)

## 8. All-review predictions and test status

In [ ]:
all_X=vectorizer.transform(df.clean_text)
df['prediction']=np.where(classifier.predict(all_X)==1,'Positive','Negative')
df['positive_probability']=classifier.predict_proba(all_X)[:,1]
df['set']= 'Train'
df.loc[test_ids,'set']='Test'
display(df[['review','sentiment','prediction','positive_probability','set']].head(12))
print('Important: predictions include training rows; model accuracy is measured on test rows only.')

## 9. Extract, refine, and categorize movie-review tags

In [ ]:
# Tags come from the original (unstemmed) review texts for readable words/phrases.
tag_vec=TfidfVectorizer(stop_words='english',ngram_range=(1,2),min_df=2,max_features=3000)
tag_matrix=tag_vec.fit_transform(df.review.str.lower())
tag_names=np.array(tag_vec.get_feature_names_out())
tag_scores=np.asarray(tag_matrix.mean(axis=0)).ravel()
exclude={'movie','film','good','bad','great','just','really','like','don','did','ve','ll','wasn','isn','doesn','didn','watch','seen','time','one','make','made'}
CATEGORY_KEYWORDS={
 'Story & Plot': {'story','plot','script','ending','twist','scene','storyline','predictable','narrative'},
 'Acting & Characters': {'acting','actor','actress','performance','cast','character','characters','role','dialogue'},
 'Direction & Production': {'director','directing','direction','cinematography','camera','effects','editing','visual','production'},
 'Sound & Music': {'music','sound','soundtrack','song','score','audio'},
 'Audience Experience': {'funny','boring','interesting','entertaining','enjoyable','enjoyed','waste','recommend','disappointing','thrilling','hilarious'}
}
def category(t):
    words=set(t.split())
    for group,keys in CATEGORY_KEYWORDS.items():
        if words & keys:return group
    return 'General opinion / other'
ranked=pd.DataFrame({'tag':tag_names,'mean_tfidf':tag_scores}).sort_values('mean_tfidf',ascending=False)
ranked=ranked[ranked.tag.map(lambda t: len(t)>=4 and t not in exclude and not any(w.isdigit() for w in t.split()))].copy()
ranked['category']=ranked.tag.map(category)
tags=ranked.head(40).reset_index(drop=True)
print('Candidate features:',len(tag_names),'Refined candidates:',len(ranked))
display(tags.head(25))
fig,ax=plt.subplots(figsize=(9,6));top=tags.head(20).iloc[::-1]
ax.barh(top.tag,top.mean_tfidf,color='#648f77');ax.set(title='Top 20 refined IMDb review tags',xlabel='Mean TF-IDF weight')
plt.tight_layout();fig.savefig(OUT/'06_tags.png',dpi=180);display(fig);plt.close(fig)
fig,ax=plt.subplots(figsize=(11,5))
wc=WordCloud(width=1100,height=520,background_color='white',colormap='viridis',random_state=42).generate_from_frequencies(dict(zip(tags.tag,tags.mean_tfidf)))
ax.imshow(wc,interpolation='bilinear');ax.axis('off');ax.set_title('IMDb movie review tag cloud')
plt.tight_layout();fig.savefig(OUT/'07_tag_cloud.png',dpi=180);display(fig);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4))
cc=tags.category.value_counts();ax.barh(cc.index[::-1],cc.values[::-1],color='#698cb0')
ax.set(title='Categories among the top 40 extracted tags',xlabel='Number of tags (not reviews)')
plt.tight_layout();fig.savefig(OUT/'08_categories.png',dpi=180);display(fig);plt.close(fig)

## 10. Export results and numerical evidence

In [ ]:
df.to_csv(OUT/'imdb_review_predictions.csv',index=False)
bow_df.to_csv(OUT/'bow_terms.csv',index=False)
tfidf_df.to_csv(OUT/'tfidf_features.csv',index=False)
tags.to_csv(OUT/'refined_tags.csv',index=False)
metrics={'dataset':'UCI Sentiment Labelled Sentences – IMDb subset',
 'review_count':len(df),'negative_count':int((df.label==0).sum()),'positive_count':int((df.label==1).sum()),
 'empty_after_preprocessing':removed,'train_count':len(train_ids),'test_count':len(test_ids),
 'bow_shape':list(B.shape),'tfidf_shape':list(Xview.shape),'explained_variance_2d':explained,
 'test_accuracy':acc,'test_confusion_matrix':confusion_matrix(yte,pred,labels=[0,1]).tolist(),
 'classification_report':report,'tag_count':len(tags)}
(OUT/'metrics.json').write_text(json.dumps(metrics,indent=2))
print(json.dumps({k:v for k,v in metrics.items() if k!='classification_report'},indent=2))
print('Saved 8 PNG charts, 4 CSV tables and metrics.json into:',OUT)

## 11. Interactive dashboard inside Jupyter
Select a tab and adjust controls. The notebook must be run from top to bottom once before opening this dashboard. For report screenshots, capture the **overview** and every stage tab with the title visible.

In [ ]:
sections=['Overview','Preprocessing','Bag-of-Words','TF-IDF','SVD / LSA','Sentiment Evaluation','Tags & Categories','Live Prediction']
tab=widgets.Tab(layout=widgets.Layout(width='100%'))
outs=[widgets.Output(layout=widgets.Layout(border='1px solid #dddddd',padding='12px')) for _ in sections]
tab.children=outs
for i,title in enumerate(sections):tab.set_title(i,title)

with outs[0]:
    display(Markdown('### IMDb Movie Review Opinion Mining Dashboard'))
    display(Markdown(f'**Reviews:** {len(df)}  |  **Positive:** {int((df.label==1).sum())}  |  **Negative:** {int((df.label==0).sum())}  |  **Test accuracy:** {acc:.2%}'))
    display(Markdown('The percentages above use **official labels**; the test accuracy is measured on held-out samples.'))
    display(widgets.Image(value=(OUT/'01_distribution.png').read_bytes(),format='png',layout=widgets.Layout(width='700px')))
    display(df[['review','sentiment']].head(10))
with outs[1]:
    display(Markdown('### Text cleaning examples'))
    display(df[['review','clean_text','sentiment']].head(18))
    display(Markdown(f'Empty rows removed after cleaning: **{removed}**. Negation words are retained.'))
with outs[2]:
    display(Markdown('### Bag-of-Words analysis'))
    display(Markdown(f'Matrix shape: **{B.shape}**'))
    display(widgets.Image(value=(OUT/'02_bow.png').read_bytes(),format='png',layout=widgets.Layout(width='800px')))
    display(bow_df.head(20))
with outs[3]:
    display(Markdown('### TF-IDF feature extraction'))
    display(Markdown(f'Descriptive matrix shape: **{Xview.shape}**'))
    display(widgets.Image(value=(OUT/'03_tfidf.png').read_bytes(),format='png',layout=widgets.Layout(width='800px')))
    display(tfidf_df.head(20))
with outs[4]:
    display(Markdown('### Two-dimensional SVD projection'))
    display(Markdown(f'Explained variance (2 components): **{explained:.2%}**. This plot is exploratory, not proof of clustering.'))
    display(widgets.Image(value=(OUT/'04_svd.png').read_bytes(),format='png',layout=widgets.Layout(width='800px')))
with outs[5]:
    display(Markdown('### Logistic Regression performance'))
    display(Markdown(f'**Train:** {len(train_ids)}  |  **Test:** {len(test_ids)}  |  **Held-out test accuracy:** {acc:.2%}'))
    display(pd.DataFrame(report).transpose().round(3))
    display(widgets.Image(value=(OUT/'05_confusion.png').read_bytes(),format='png',layout=widgets.Layout(width='700px')))
with outs[6]:
    display(Markdown('### Ranked, refined, and enriched tags'))
    display(widgets.Image(value=(OUT/'06_tags.png').read_bytes(),format='png',layout=widgets.Layout(width='750px')))
    display(widgets.Image(value=(OUT/'07_tag_cloud.png').read_bytes(),format='png',layout=widgets.Layout(width='750px')))
    display(widgets.Image(value=(OUT/'08_categories.png').read_bytes(),format='png',layout=widgets.Layout(width='750px')))
    display(tags.head(25))
with outs[7]:
    display(Markdown('### Try your own IMDb review'))
    prompt=widgets.Textarea(value='The acting was amazing and the story was exciting.',description='Review:',layout=widgets.Layout(width='95%',height='100px'))
    run=widgets.Button(description='Predict sentiment',button_style='info')
    result=widgets.Output()
    def predict_review(_):
        with result:
            clear_output(wait=True)
            cleaned=clean_text(prompt.value)
            if not cleaned:
                print('Please enter a review containing words.')
                return
            prob=float(classifier.predict_proba(vectorizer.transform([cleaned]))[0,1])
            print('Prediction:', 'Positive' if prob>=.5 else 'Negative')
            print(f'Positive probability: {prob:.1%}')
            print('This is a model estimate, not a ground-truth label.')
    run.on_click(predict_review)
    display(prompt,run,result)
display(tab)

## 12. Screenshot checklist for the final report
Capture screenshots after **Run All**. Screenshots should show the actual table/chart output, not only code.

1. Dataset loaded + first 10 reviews + class counts.
2. Cleaning table (raw vs cleaned text) + empty-row count.
3. Sentiment distribution chart.
4. Bag-of-Words matrix dimensions + top terms table + plot.
5. TF-IDF dimensions + top features table + plot.
6. SVD explained variance text + scatterplot.
7. Train/test count + accuracy + classification report.
8. Confusion matrix.
9. Tag ranking table + tag bar chart.
10. Tag word cloud + tag categories chart.
11. Dashboard Overview tab.
12. Dashboard Sentiment Evaluation and Tags tabs.
13. Live Prediction tab after pressing **Predict sentiment** for a sample review.

Also send `outputs/metrics.json` and `outputs/refined_tags.csv` so the report can use exact, verified values. **Do not use the example report's figures as our results.**